In [2]:
import os
print(os.getcwd())


/Users/matt/Desktop/Patent Litigation Analysis/notebooks


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [4]:
from IPython.display import display

In [39]:
pacer_cases = pd.read_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v2.csv")
attorneys = pd.read_csv("../data/processed/attorneys_cleaned.csv")
documents = pd.read_csv("../data/processed/documents_cleaned.csv")
names = pd.read_csv("../data/processed/names_cleaned.csv")
cases = pd.read_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases_processed_v3.csv")

/var/folders/51/1bmpv68x1h3gvxqw6fktjqh80000gn/T/ipykernel_62636/203661647.py:3: DtypeWarning: Columns (0: doc_number, 1: short_description) have mixed types. Specify dtype option on import or set low_memory=False.
  documents = pd.read_csv("../data/processed/documents_cleaned.csv")


In [41]:
print("Cases Shape")
print(cases.shape)
print("Cases Columns")
print(cases.columns.tolist())
print("Cases Data Types")
print(cases.dtypes)
print("Cases Head")
print(cases.head(10))

Cases Shape
(74629, 21)
Cases Columns
['case_row_id', 'case_number', 'pacer_id', 'case_name', 'court_name', 'assigned_to', 'referred_to', 'case_cause', 'jurisdictional_basis', 'demand', 'jury_demand', 'lead_case', 'related_case', 'settlement', 'date_filed', 'date_closed', 'date_last_filed', 'region', 'case_topic', 'case_number_clean', 'case_number_no_prefix']
Cases Data Types
case_row_id                int64
case_number                  str
pacer_id                 float64
case_name                    str
court_name                   str
assigned_to                  str
referred_to                  str
case_cause                   str
jurisdictional_basis         str
demand                       str
jury_demand                  str
lead_case                    str
related_case                 str
settlement                   str
date_filed                   str
date_closed                  str
date_last_filed              str
region                       str
case_topic                 

In [42]:
cases.isnull().sum()
cases.isnull().mean().sort_values(ascending=False)
cases[cases.isnull().any(axis=1)]


,case_row_id,case_number,pacer_id,case_name,court_name,assigned_to,referred_to,case_cause,jurisdictional_basis,demand,...,lead_case,related_case,settlement,date_filed,date_closed,date_last_filed,region,case_topic,case_number_clean,case_number_no_prefix
0,74674,9:2015-mc-80452,460803.0,"Mobile Telecommunications Technologies, LLC v....",UNITED STATES DISTRICT COURT SOUTHERN DISTRICT...,Judge William J. Zloch,NaN,28:1331 Federal Question,Federal Question,NaN,...,NaN,NaN,NaN,4/7/15,4/22/15,NaN,Florida,Federal Question,9:2015-mc-80452,2015-mc-80452
1,74673,9:2015-cv-81758,NaN,"Voltstar Technologies, Inc. v. L & L Wings, Inc.",Southern District of Florida (West Palm Beach),Judge James I. Cohn,Magistrate Judge Barry S. Seltzer,35:0271 Patent Infringement,Federal Question,Plaintiff,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Patent,9:2015-cv-81758,2015-cv-81758
2,74672,9:2015-cv-81753,NaN,"Allied Mineral Products, Inc. v. OSMI, Inc. et al",Southern District of Florida (West Palm Beach),Judge Kenneth A. Marra,NaN,28:2201 Declaratory Judgment,Federal Question,Plaintiff,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Declaratory Judgment,9:2015-cv-81753,2015-cv-81753
3,74671,9:2015-cv-81703,NaN,"Shipping and Transit, LLC v. Trans-Expedite, Inc.",Southern District of Florida (West Palm Beach),Judge Beth Bloom,NaN,15:1126 Patent Infringement,Federal Question,Plaintiff,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Patent,9:2015-cv-81703,2015-cv-81703
4,74670,9:2015-cv-81692,NaN,NaN,Southern District of Florida (West Palm Beach),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,NaN,9:2015-cv-81692,2015-cv-81692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74624,5,0:1984-cv-06726,NaN,"Monaco Del, et al v. Atlantic Federal S &, et al",Southern District of Florida (Ft Lauderdale),Senior Judge William M. Hoeveler,NaN,15:1126 Patent Infringement,Federal Question,Plaintiff,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Patent,0:1984-cv-06726,1984-cv-06726
74625,4,0:1984-cv-06456,NaN,"Monte Carlo Hairpiec, et al v. On-Rite Hairpie...",Southern District of Florida (Ft Lauderdale),Senior Judge Kenneth L. Ryskamp,NaN,28:1338,Federal Question,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Other,0:1984-cv-06456,1984-cv-06456
74626,3,0:1983-cv-06860,NaN,Cornwall v. U. S. Construction,Southern District of Florida (Ft Lauderdale),"Judge Jose A. Gonzalez, Jr",NaN,35:0145 Patent Infringement,Federal Question,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,Patent,0:1983-cv-06860,1983-cv-06860
74627,2,0:1981-cv-06112,NaN,v.,Southern District of Florida (Ft Lauderdale),Senior Judge Lenore C. Nesbitt,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Florida,NaN,0:1981-cv-06112,1981-cv-06112


In [8]:
import re

cases['case_number_clean'] = (
    cases['case_number_clean']
    .str.replace('CIVIL DOCKET FOR CASE #', '', regex=False)
    .str.strip()
)

def standardize_case_number(case_num):
    if pd.isna(case_num):
        return case_num

    match = re.match(r'(\d+):(\d{2})-(\w+)-(\d+)(-\w+)?', case_num)

    if match:
        prefix = match.group(1)
        year = match.group(2)
        case_type = match.group(3).lower()
        number = match.group(4)

        year = '19' + year if int(year) >= 50 else '20' + year

        return f"{prefix}:{year}-{case_type}-{number}"

    return case_num

cases['case_number_clean'] = (
    cases['case_number_clean']
    .apply(standardize_case_number)
)


In [47]:
print("Case Number fix")
print(cases['case_number_clean'].head(30))

Case Number fix
0     9:2015-mc-80452
1     9:2015-cv-81758
2     9:2015-cv-81753
3     9:2015-cv-81703
4     9:2015-cv-81692
5     9:2015-cv-81609
6     9:2015-cv-81608
7     9:2015-cv-81607
8     9:2015-cv-81596
9     9:2015-cv-81582
10    9:2015-cv-81550
11    9:2015-cv-81529
12    9:2015-cv-81360
13    9:2015-cv-81359
14    9:2015-cv-81358
15    9:2015-cv-81190
16    9:2015-cv-81086
17    9:2015-cv-81085
18    9:2015-cv-81083
19    9:2015-cv-81076
20    9:2015-cv-81035
21    9:2015-cv-81034
22    9:2015-cv-81032
23    9:2015-cv-81031
24    9:2015-cv-81030
25    9:2015-cv-81004
26    9:2015-cv-80916
27    9:2015-cv-80906
28    9:2015-cv-80882
29    9:2015-cv-80881
Name: case_number_clean, dtype: str


In [48]:
pacer_cases['case_number_clean'] = pacer_cases['case_number_clean'].str.strip()

matched = cases['case_number_clean'].isin(pacer_cases['case_number_clean'])
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Total cases rows:", len(cases))

cases[~matched][['case_number_clean', 'case_number_clean']].head(20)

Matched: 74603
Unmatched: 26
Total cases rows: 74629


,case_number_clean,case_number_clean
10195,4:2015-cv-05489,4:2015-cv-05489
10196,4:2015-cv-05486,4:2015-cv-05486
10197,4:2015-cv-05482,4:2015-cv-05482
10198,4:2015-cv-05480,4:2015-cv-05480
10199,4:2015-cv-05479,4:2015-cv-05479
10200,3:2015-cv-05477,3:2015-cv-05477
10210,3:2015-cv-04428,3:2015-cv-04428
13952,3:2015-cv-05793,3:2015-cv-05793
13955,5:2015-cv-05438,5:2015-cv-05438
15450,3:2008-cv-01217,3:2008-cv-01217


In [49]:
unmatched = cases[~matched][['case_number_clean', 'case_number_clean']]

pd.set_option('display.max_rows', 30)
display(unmatched)

,case_number,case_number_clean
10195,4:2015-cv-05489,4:2015-cv-05489
10196,4:2015-cv-05486,4:2015-cv-05486
10197,4:2015-cv-05482,4:2015-cv-05482
10198,4:2015-cv-05480,4:2015-cv-05480
10199,4:2015-cv-05479,4:2015-cv-05479
10200,3:2015-cv-05477,3:2015-cv-05477
10210,3:2015-cv-04428,3:2015-cv-04428
13952,3:2015-cv-05793,3:2015-cv-05793
13955,5:2015-cv-05438,5:2015-cv-05438
15450,3:2008-cv-01217,3:2008-cv-01217


In [ ]:
# Look at the structure of the unmatched case_number_clean values
for val in unmatched['case_number_clean'].tolist():
    print(repr(val))i

'4:2015-cv-05489'
'4:2015-cv-05486'
'4:2015-cv-05482'
'4:2015-cv-05480'
'4:2015-cv-05479'
'3:2015-cv-05477'
'3:2015-cv-04428'
'3:2015-cv-05793'
'5:2015-cv-05438'
'3:2008-cv-01217'
'3:2008-cv-00995'
'3:2007-cv-04125'
'3:2004-cv-04922'
'4:2015-cv-05490'
'4:2015-cv-05488'
'4:2015-cv-05487'
'4:2015-cv-05485'
'4:2015-cv-05481'
'4:2015-cv-05478'
'5:2015-cv-05443'
'5:2015-cv-05440'
'5:2015-cv-04214'
'4:2014-cv-04908'
'2:2014-cv-07080'
'2:2013-cv-07766'


In [13]:
print("Cases dtype:", cases['case_number_clean'].dtype)
print("Pacer dtype:", pacer_cases['case_number_clean'].dtype)

Cases dtype: str
Pacer dtype: str


In [ ]:
cases['case_number_clean'] = cases['case_number_clean'].str.replace(r'[^a-zA-Z0-9\-:]', '', regex=True)
pacer_cases['case_number'] = pacer_cases['case_number'].str.replace(r'[^a-zA-Z0-9\-:]', '', regex=True)

In [15]:
matched = cases['case_number_clean'].isin(pacer_cases['case_number'])
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Total cases rows:", len(cases))

Matched: 74604
Unmatched: 25
Total cases rows: 74629


In [16]:
display(unmatched)

,case_number,case_number_clean
10195,4:2015-cv-05489,4:2015-cv-05489
10196,4:2015-cv-05486,4:2015-cv-05486
10197,4:2015-cv-05482,4:2015-cv-05482
10198,4:2015-cv-05480,4:2015-cv-05480
10199,4:2015-cv-05479,4:2015-cv-05479
10200,3:2015-cv-05477,3:2015-cv-05477
10210,3:2015-cv-04428,3:2015-cv-04428
13952,3:2015-cv-05793,3:2015-cv-05793
13955,5:2015-cv-05438,5:2015-cv-05438
15450,3:2008-cv-01217,3:2008-cv-01217


In [17]:
cases['case_number_no_prefix'] = cases['case_number_clean'].str.split(':').str[1]
pacer_cases['case_number_no_prefix'] = pacer_cases['case_number'].str.split(':').str[1]

matched_no_prefix = cases['case_number_no_prefix'].isin(pacer_cases['case_number_no_prefix'])
print("Matched without prefix:", matched_no_prefix.sum())
print("Unmatched without prefix:", (~matched_no_prefix).sum())

Matched without prefix: 74629
Unmatched without prefix: 0


In [18]:
cases['case_number_no_prefix'] = cases['case_number_no_prefix'].str.strip()
pacer_cases['case_number_no_prefix'] = pacer_cases['case_number_no_prefix'].str.strip()

pacer_mapping = pacer_cases.set_index('case_number_no_prefix')['case_number'].to_dict()

cases['case_number_clean'] = cases['case_number_no_prefix'].map(pacer_mapping).fillna(cases['case_number_clean'])

matched = cases['case_number_clean'].isin(pacer_cases['case_number'])
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Total cases rows:", len(cases))

unmatched_final = cases[~matched][['case_number', 'case_number_clean']]
display(unmatched_final)

Matched: 74629
Unmatched: 0
Total cases rows: 74629


,case_number,case_number_clean


In [19]:
row = cases[cases['case_number_clean'].str.contains('022941', na=False)]
print(repr(row['case_number_clean'].iloc[0]))
print(repr(row['case_number_no_prefix'].iloc[0]))

IndexError: single positional indexer is out-of-bounds

In [ ]:
unmatched_final = cases[~matched][['case_number', 'case_number_clean', 'case_number_no_prefix']]
display(unmatched_final)

for idx, row in unmatched_final.iterrows():
    print(f"Row {idx}:")
    print(f"  case_number repr: {repr(row['case_number'])}")
    print(f"  case_number_clean repr: {repr(row['case_number_clean'])}")
    print(f"  case_number_no_prefix repr: {repr(row['case_number_no_prefix'])}")
    print()

,case_number,case_number_clean,case_number_no_prefix
12825,1:2011-md-02294,1:2011-md-02294,2011-md-02294


Row 12825:
  case_number repr: '1:2011-md-02294'
  case_number_clean repr: '1:2011-md-02294'
  case_number_no_prefix repr: '2011-md-02294'



In [ ]:
matched = cases['case_number_clean'].isin(pacer_cases['case_number'])
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())

Matched: 74629
Unmatched: 0


In [ ]:
cases.loc[cases['case_number'] == '1:2011-md-02294', 'case_number_clean'] = '1:2011-md-02241'

In [ ]:
print(pacer_cases[pacer_cases['case_number'].str.contains('2011-md', na=False)]['case_number'].tolist())

['0:2011-md-02249', '1:2011-md-02241']


In [ ]:
print(cases[cases['case_number_clean'].str.contains('2011-md', na=False)]['case_number_clean'].tolist())

['0:2011-md-02249', '1:2011-md-02241', '1:2011-md-02294']


In [ ]:
pacer_cases.to_csv("pacer_cases_processed_v2.csv", index=False)

In [ ]:
cases_processed = pd.read_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases_processed_v2.csv")
pacer_cases_processed = pd.read_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v2.csv")


In [ ]:
matched = cases_processed['case_number_clean'].isin(pacer_cases_processed['case_number'])
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Total rows:", len(cases_processed))

Matched: 74629
Unmatched: 0
Total rows: 74629


In [20]:
attorneys.head(5)


,case_row_id,case_number,party_row_count,party_type,attorney_row_count,name,contactinfo,position,terminated_date,is_lead_attorney,is_inactive,firm,address,region
0,14,0:92-cv-00398-MJP,40,Plaintiff,1,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
1,14,0:92-cv-00398-MJP,41,Plaintiff,2,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
2,14,0:92-cv-00398-MJP,42,Plaintiff,3,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
3,14,0:92-cv-00398-MJP,43,Plaintiff,4,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
4,14,0:92-cv-00398-MJP,44,Plaintiff,5,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina


In [21]:
import re
import pandas as pd

attorneys['case_number_clean'] = (
    attorneys['case_number']
    .str.replace('CIVIL DOCKET FOR CASE #', '', regex=False)
    .str.strip()
)

def standardize_case_number(case_num):
    if pd.isna(case_num):
        return case_num

    match = re.match(r'(\d+):(\d{2})-(\w+)-(\d+)(-\w+)?', case_num)

    if match:
        prefix = match.group(1)
        year = match.group(2)
        case_type = match.group(3).lower()
        number = match.group(4)

        year = '19' + year if int(year) >= 50 else '20' + year

        return f"{prefix}:{year}-{case_type}-{number}"

    return case_num

attorneys['case_number_clean'] = (
    attorneys['case_number_clean']
    .apply(standardize_case_number)
)

In [22]:
attorneys['case_number_clean'] = attorneys['case_number_clean'].str.replace(
    r'(\d+:\d{4}-)\w+(-\d+)',
    lambda m: m.group(1) + 'cv' + m.group(2),
    regex=True
)

In [23]:
cases_keys = set(cases['case_number_clean'].dropna())

attorneys['match'] = attorneys['case_number_clean'].isin(cases_keys)

total = len(attorneys)
matched = attorneys['match'].sum()
unmatched = total - matched

print(f"Total attorney rows:   {total}")
print(f"Matched:               {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:             {unmatched} ({unmatched/total*100:.1f}%)")

unmatched_vals = attorneys.loc[~attorneys['match'], 'case_number_clean'].unique()
print(f"\nUnique unmatched case numbers ({len(unmatched_vals)}):")
print(unmatched_vals[:50])

Total attorney rows:   1223417
Matched:               861395 (70.4%)
Unmatched:             362022 (29.6%)

Unique unmatched case numbers (16507):
<StringArray>
['0:2008-cv-60570', '0:2010-cv-61894', '0:2011-cv-60082', '0:2015-cv-62581',
 '1:1985-cv-00014', '1:1992-cv-00398', '1:1992-cv-00014', '1:1993-cv-06231',
 '1:1994-cv-06756', '1:1995-cv-00082', '1:1996-cv-02293', '1:1996-cv-00032',
 '1:1996-cv-00098', '1:1996-cv-00064', '1:1997-cv-00003', '1:1997-cv-00170',
 '1:1997-cv-00203', '1:1997-cv-00740', '1:1997-cv-01732', '1:1997-cv-01756',
 '1:1997-cv-02355', '1:1997-cv-02404', '1:1997-cv-02459', '1:1997-cv-07597',
 '1:1997-cv-00104', '1:1997-cv-00115', '1:1998-cv-00551', '1:1998-cv-01112',
 '1:1998-cv-01494', '1:1998-cv-02113', '1:1998-cv-05942', '1:1998-cv-00001',
 '1:1998-cv-01228', '1:1999-cv-00061', '1:1999-cv-00133', '1:1999-cv-00261',
 '1:1999-cv-00291', '1:1999-cv-00294', '1:1999-cv-00369', '1:1999-cv-00501',
 '1:1999-cv-00752', '1:1999-cv-00773', '1:1999-cv-06465', '1:1999-cv-

In [24]:
unmatched_in_cases = [x for x in unmatched_vals if x in cases_keys]
truly_unmatched = [x for x in unmatched_vals if x not in cases_keys]

print(f"Unmatched vals that DO exist in cases: {len(unmatched_in_cases)}")
print(f"Unmatched vals that truly don't exist in cases: {len(truly_unmatched)}")
print(truly_unmatched[:50])


Unmatched vals that DO exist in cases: 0
Unmatched vals that truly don't exist in cases: 16507
['0:2008-cv-60570', '0:2010-cv-61894', '0:2011-cv-60082', '0:2015-cv-62581', '1:1985-cv-00014', '1:1992-cv-00398', '1:1992-cv-00014', '1:1993-cv-06231', '1:1994-cv-06756', '1:1995-cv-00082', '1:1996-cv-02293', '1:1996-cv-00032', '1:1996-cv-00098', '1:1996-cv-00064', '1:1997-cv-00003', '1:1997-cv-00170', '1:1997-cv-00203', '1:1997-cv-00740', '1:1997-cv-01732', '1:1997-cv-01756', '1:1997-cv-02355', '1:1997-cv-02404', '1:1997-cv-02459', '1:1997-cv-07597', '1:1997-cv-00104', '1:1997-cv-00115', '1:1998-cv-00551', '1:1998-cv-01112', '1:1998-cv-01494', '1:1998-cv-02113', '1:1998-cv-05942', '1:1998-cv-00001', '1:1998-cv-01228', '1:1999-cv-00061', '1:1999-cv-00133', '1:1999-cv-00261', '1:1999-cv-00291', '1:1999-cv-00294', '1:1999-cv-00369', '1:1999-cv-00501', '1:1999-cv-00752', '1:1999-cv-00773', '1:1999-cv-06465', '1:1999-cv-00006', '1:1999-cv-00032', '1:2000-cv-00295', '1:2000-cv-00464', '1:2000-cv-

In [25]:
cases_keys = set(cases['case_number_clean'].dropna())

attorneys['match'] = attorneys['case_number_clean'].isin(cases_keys)

total = len(attorneys)
matched = attorneys['match'].sum()
unmatched = total - matched

print(f"Total attorney rows:   {total}")
print(f"Matched:               {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:             {unmatched} ({unmatched/total*100:.1f}%)")

unmatched_vals = attorneys.loc[~attorneys['match'], 'case_number_clean'].unique()
print(f"\nUnique unmatched case numbers ({len(unmatched_vals)}):")
print(unmatched_vals[:50])

Total attorney rows:   1223417
Matched:               861395 (70.4%)
Unmatched:             362022 (29.6%)

Unique unmatched case numbers (16507):
<StringArray>
['0:2008-cv-60570', '0:2010-cv-61894', '0:2011-cv-60082', '0:2015-cv-62581',
 '1:1985-cv-00014', '1:1992-cv-00398', '1:1992-cv-00014', '1:1993-cv-06231',
 '1:1994-cv-06756', '1:1995-cv-00082', '1:1996-cv-02293', '1:1996-cv-00032',
 '1:1996-cv-00098', '1:1996-cv-00064', '1:1997-cv-00003', '1:1997-cv-00170',
 '1:1997-cv-00203', '1:1997-cv-00740', '1:1997-cv-01732', '1:1997-cv-01756',
 '1:1997-cv-02355', '1:1997-cv-02404', '1:1997-cv-02459', '1:1997-cv-07597',
 '1:1997-cv-00104', '1:1997-cv-00115', '1:1998-cv-00551', '1:1998-cv-01112',
 '1:1998-cv-01494', '1:1998-cv-02113', '1:1998-cv-05942', '1:1998-cv-00001',
 '1:1998-cv-01228', '1:1999-cv-00061', '1:1999-cv-00133', '1:1999-cv-00261',
 '1:1999-cv-00291', '1:1999-cv-00294', '1:1999-cv-00369', '1:1999-cv-00501',
 '1:1999-cv-00752', '1:1999-cv-00773', '1:1999-cv-06465', '1:1999-cv-

In [26]:
attorneys.loc[attorneys['case_number_clean'] == '10-pos-1', 'case_number_clean'] = '2:2012-cv-05711'

In [27]:
attorneys.loc[attorneys['case_number_clean'] == '1-Jan-70', 'case_number_clean'] = '3:2009-cv-00199'

In [28]:
attorneys.loc[attorneys['case_number_clean'] == '01-Jan-70', 'case_number_clean'] = '3:2009-cv-00199'

In [29]:
attorneys.loc[attorneys['case_number_clean'] == '01-Jan-1970', 'case_number_clean'] = '3:2009-cv-00199'

In [30]:
cases_keys = set(cases['case_number_clean'].dropna())

attorneys['match'] = attorneys['case_number_clean'].isin(cases_keys)

total = len(attorneys)
matched = attorneys['match'].sum()
unmatched = total - matched

print(f"Total attorney rows:   {total}")
print(f"Matched:               {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:             {unmatched} ({unmatched/total*100:.1f}%)")

unmatched_vals = attorneys.loc[~attorneys['match'], 'case_number_clean'].unique()
print(f"\nUnique unmatched case numbers ({len(unmatched_vals)}):")
print(unmatched_vals[:50])

Total attorney rows:   1223417
Matched:               861475 (70.4%)
Unmatched:             361942 (29.6%)

Unique unmatched case numbers (16505):
<StringArray>
['0:2008-cv-60570', '0:2010-cv-61894', '0:2011-cv-60082', '0:2015-cv-62581',
 '1:1985-cv-00014', '1:1992-cv-00398', '1:1992-cv-00014', '1:1993-cv-06231',
 '1:1994-cv-06756', '1:1995-cv-00082', '1:1996-cv-02293', '1:1996-cv-00032',
 '1:1996-cv-00098', '1:1996-cv-00064', '1:1997-cv-00003', '1:1997-cv-00170',
 '1:1997-cv-00203', '1:1997-cv-00740', '1:1997-cv-01732', '1:1997-cv-01756',
 '1:1997-cv-02355', '1:1997-cv-02404', '1:1997-cv-02459', '1:1997-cv-07597',
 '1:1997-cv-00104', '1:1997-cv-00115', '1:1998-cv-00551', '1:1998-cv-01112',
 '1:1998-cv-01494', '1:1998-cv-02113', '1:1998-cv-05942', '1:1998-cv-00001',
 '1:1998-cv-01228', '1:1999-cv-00061', '1:1999-cv-00133', '1:1999-cv-00261',
 '1:1999-cv-00291', '1:1999-cv-00294', '1:1999-cv-00369', '1:1999-cv-00501',
 '1:1999-cv-00752', '1:1999-cv-00773', '1:1999-cv-06465', '1:1999-cv-

In [31]:
pacer_cases = pacer_cases[~pacer_cases['case_number_clean'].str.contains('bfontcolorre|fontcolor|font', na=False, regex=True)]

In [32]:
pacer_cases = pacer_cases[~pacer_cases['case_number_clean'].str.startswith('font', na=False)]

In [33]:
pacer_cases = pacer_cases[~pacer_cases['case_number_clean'].str.startswith(('font', 'b'), na=False)]

In [34]:
pacer_cases['case_number_clean'] = pacer_cases['case_number_clean'].str.replace(
    r'(\d+:\d{4}-)\w+(-\d+)',
    lambda m: m.group(1) + 'cv' + m.group(2),
    regex=True
)

In [ ]:
cases['case_number_clean'] = cases['case_number_clean'].str.replace(
    r'(\d+:\d{4}-)\w+(-\d+)',
    lambda m: m.group(1) + 'cv' + m.group(2),
    regex=True
)

In [35]:
cases_keys = set(cases['case_number_clean'].dropna())

pacer_cases['match'] = pacer_cases['case_number_clean'].isin(cases_keys)

total = len(pacer_cases)
matched = pacer_cases['match'].sum()
unmatched = total - matched

print(f"Total pacer_cases rows: {total}")
print(f"Matched:                {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:              {unmatched} ({unmatched/total*100:.1f}%)")

Total pacer_cases rows: 74851
Matched:                56546 (75.5%)
Unmatched:              18305 (24.5%)


In [36]:
unmatched_vals = pacer_cases.loc[~pacer_cases['match'], 'case_number_clean'].unique()
print(f"Unique unmatched case numbers ({len(unmatched_vals)}):")
print(unmatched_vals)

Unique unmatched case numbers (17049):
<StringArray>
['9:2015-cv-80452', '9:2013-cv-80769', '9:2013-cv-80395', '9:2013-cv-80113',
 '9:2013-cv-00197', '9:2013-cv-00102', '9:2013-cv-00100', '9:2013-cv-00099',
 '9:2013-cv-00092', '9:2013-cv-00091',
 ...
 '0:2015-cv-62581', '0:2015-cv-62578', '0:2014-cv-62367', '0:2013-cv-61681',
 '0:2011-cv-02249', '0:2011-cv-60082', '0:2011-cv-60079', '0:2011-cv-60363',
 '0:2010-cv-61894', '0:2008-cv-60570']
Length: 17049, dtype: str


In [37]:
cases_keys = set(cases['case_number_clean'].dropna())

attorneys['match'] = attorneys['case_number_clean'].isin(cases_keys)

total = len(attorneys)
matched = attorneys['match'].sum()
unmatched = total - matched

print(f"Total attorney rows:   {total}")
print(f"Matched:               {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:             {unmatched} ({unmatched/total*100:.1f}%)")

unmatched_vals = attorneys.loc[~attorneys['match'], 'case_number_clean'].unique()
truly_unmatched = [x for x in unmatched_vals if x not in cases_keys]

print(f"\nUnique unmatched case numbers: {len(unmatched_vals)}")
print(f"Truly not in cases df:         {len(truly_unmatched)}")
print(truly_unmatched[:50])

Total attorney rows:   1223417
Matched:               861475 (70.4%)
Unmatched:             361942 (29.6%)

Unique unmatched case numbers: 16505
Truly not in cases df:         16505
['0:2008-cv-60570', '0:2010-cv-61894', '0:2011-cv-60082', '0:2015-cv-62581', '1:1985-cv-00014', '1:1992-cv-00398', '1:1992-cv-00014', '1:1993-cv-06231', '1:1994-cv-06756', '1:1995-cv-00082', '1:1996-cv-02293', '1:1996-cv-00032', '1:1996-cv-00098', '1:1996-cv-00064', '1:1997-cv-00003', '1:1997-cv-00170', '1:1997-cv-00203', '1:1997-cv-00740', '1:1997-cv-01732', '1:1997-cv-01756', '1:1997-cv-02355', '1:1997-cv-02404', '1:1997-cv-02459', '1:1997-cv-07597', '1:1997-cv-00104', '1:1997-cv-00115', '1:1998-cv-00551', '1:1998-cv-01112', '1:1998-cv-01494', '1:1998-cv-02113', '1:1998-cv-05942', '1:1998-cv-00001', '1:1998-cv-01228', '1:1999-cv-00061', '1:1999-cv-00133', '1:1999-cv-00261', '1:1999-cv-00291', '1:1999-cv-00294', '1:1999-cv-00369', '1:1999-cv-00501', '1:1999-cv-00752', '1:1999-cv-00773', '1:1999-cv-06465', 

NameError: name 'python' is not defined

In [38]:
attorneys[attorneys['case_number_clean'] == '1:2011-cv-02294'].head()

,case_row_id,case_number,party_row_count,party_type,attorney_row_count,name,contactinfo,position,terminated_date,is_lead_attorney,is_inactive,firm,address,region,case_number_clean,match
282413,19025,1:11-md-02294,105763,Counter Defendant,282454,"William Ellsworth Davis , III","The Davis Firm PC; 111 W Tyler St; Longview, T...",TERMINATED: 10/23/2014; ATTORNEY TO BE NOTICED,2014-10-23,False,False,The Davis Firm PC,"111 W Tyler St, Longview, TX 75601",Texas,1:2011-cv-02294,False
282414,19025,1:11-md-02294,105764,Counter Claimant,282455,Mark Philip Wine,Orrick Herrington and Sutcliffe LLP; 2050 Main...,LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Orrick Herrington and Sutcliffe LLP,"2050 Main St Ste 1100, Irvine, CA 92614",California,1:2011-cv-02294,False
282415,19025,1:11-md-02294,105764,Counter Claimant,282456,Benjamin S Lin,Orrick Herrington and Sutcliffe LLP; 2050 Main...,ATTORNEY TO BE NOTICED,NaN,False,False,Orrick Herrington and Sutcliffe LLP,"2050 Main St Ste 1100, Irvine, CA 92614",California,1:2011-cv-02294,False
282416,19025,1:11-md-02294,105764,Counter Claimant,282457,Thomas J Gray,Orrick Herrington and Sutcliffe LLP; 2050 Main...,ATTORNEY TO BE NOTICED,NaN,False,False,Orrick Herrington and Sutcliffe LLP,"2050 Main St Ste 1000, Irvine, CA 92614",California,1:2011-cv-02294,False
282417,19025,1:11-md-02294,105765,Counter Claimant,282458,Mark Philip Wine,Orrick Herrington and Sutcliffe LLP; 2050 Main...,LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Orrick Herrington and Sutcliffe LLP,"2050 Main St Ste 1100, Irvine, CA 92614",California,1:2011-cv-02294,False
